In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
silver_schema = "silver"
bronze_schema = "bronze"

print("=" * 80)
print("SILVER LAYER VERIFICATION")
print("=" * 80)

# List all Silver tables
silver_tables = spark.sql(f"SHOW TABLES IN {catalog}.{silver_schema}").collect()

print(f"\nTotal Silver tables created: {len(silver_tables)}\n")

for table in silver_tables:
    table_name = table.tableName
    count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.{silver_schema}.{table_name}").collect()[0][0]
    print(f"{table_name}: {count:,} rows")

# Bronze vs Silver comparison
print("\n" + "=" * 80)
print("BRONZE vs SILVER ROW COUNT COMPARISON")
print("=" * 80)

comparisons = [
    ("crm_cust_info", "customers"),
    ("crm_prd_info", "products"),
    ("crm_sales_details", "sales"),
    ("erp_cust_az12", "erp_customers"),
    ("erp_loc_a101", "locations"),
    ("erp_px_cat_g1v2", "categories")
]

print()
for bronze_table, silver_table in comparisons:
    try:
        bronze_count = spark.sql(
            f"SELECT COUNT(*) FROM {catalog}.{bronze_schema}.{bronze_table}"
        ).collect()[0][0]
        
        silver_count = spark.sql(
            f"SELECT COUNT(*) FROM {catalog}.{silver_schema}.{silver_table}"
        ).collect()[0][0]
        
        removed = bronze_count - silver_count
        percent = (removed / bronze_count * 100) if bronze_count > 0 else 0
        
        print(f"{bronze_table:<25} → {silver_table:<20}")
        print(f"  Bronze: {bronze_count:>10,} | Silver: {silver_count:>10,} | Removed: {removed:>8,} ({percent:>5.1f}%)")
    except Exception as e:
        print(f"{bronze_table:<25} → {silver_table:<20} - ERROR: {str(e)}")

print("\n" + "=" * 80)